# Patient Data Export

A notebook exporting patient data for study analyses.

## Definitions

### Imports

In [ ]:
import dataclasses
import datetime
import io
import itertools
import json
import operator
import os
import pathlib
import pprint
import re
from enum import Enum
from typing import Any, Callable, Dict, List, Optional, Sequence, Set, cast

import bson.objectid
import IPython.display
import ipywidgets
import nbformat
import pandas as pd
import pytz
import pyzipper
import scope.database.date_utils as date_utils
import scope.enums
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from scope.documents import document_set
from scope.documents.document_set import datetime_from_document
from scope.populate.data.archive import Archive

### Constants and Flags

In [ ]:
# In development, it can be helpful to sample a subset of patients.
# If DEVELOPMENT_SAMPLE_PATIENTS <= 0, process all patients.
# If DEVELOPMENT_SAMPLE_PATIENTS > 0, randomly sample DEVELOPMENT_SAMPLE_PATIENTS patients.
DEVELOPMENT_SAMPLE_PATIENTS: int = -1

# In development, it can be helpful to skip per-patient export.
# If DEVELOPMENT_EXPORT_PER_PATIENT_DOCUMENTS, include per-patient export.
DEVELOPMENT_EXPORT_PER_PATIENT_DOCUMENTS: bool = False

# In development, it can be helpful to skip documents export.
# If DEVELOPMENT_EXPORT_COMBINED_DOCUMENTS, include documents export.
DEVELOPMENT_EXPORT_COMBINED_DOCUMENTS: bool = False

### Utilities

### Utility: documentation_as_markdown

Returns a string containing markdown content recovered from a cell in this notebook.

Intended to allow including the content of markdown cells as documentation in an export.

In [ ]:
def documentation_as_markdown(documentation_name: str) -> str:
    # Load this same notebook.
    notebook = nbformat.read("patientdata.ipynb", nbformat.NO_CONVERT)

    # Go through each cell, looking for a match.
    for cell_current in notebook["cells"]:
        match = cell_current["cell_type"] == "markdown"
        if match:
            match = re.match(
                "^(#*) Documentation: ({})\\n(.*)".format(documentation_name),
                cell_current["source"],
            )

        if match:
            return cell_current["source"]

    # If no match was found, raise a ValueError.
    raise ValueError(
        "No matching documentation cell found: {}".format(documentation_name)
    )

### Utility: ExportFile

The path and contents of a file to be exported.

In [ ]:
class ExportFileType(Enum):
    BYTES = "BYTES"
    CSV = "CSV"
    EXCEL = "EXCEL"
    MARKDOWN = "MARKDOWN"


@dataclasses.dataclass(frozen=True)
class ExportFile:
    path: pathlib.Path
    type: ExportFileType
    bytes: Optional[bytes]
    text: Optional[str]

### Utility: export_file_list

In [ ]:
export_file_list: List[ExportFile] = []

### Utility: export_dataframe_as_csv

In [ ]:
def export_dataframe_as_csv(
    path: pathlib.Path,
    df: pd.DataFrame,
):
    # Extend any existing suffix.
    path = path.with_suffix(path.suffix + ".csv")

    csv_text = df.to_csv(index=False)

    export_file_list.append(
        ExportFile(
            path=path,
            type=ExportFileType.CSV,
            bytes=None,
            text=csv_text,
        )
    )

### Utility: export_dataframe_as_excel

In [ ]:
def export_dataframe_as_excel(
    path: pathlib.Path,
    df: pd.DataFrame,
):
    # Extend any existing suffix.
    path = path.with_suffix(path.suffix + ".xlsx")

    iobytes = io.BytesIO()
    # BytesIO works fine at runtime, but pandas does not list it as a valid excel_writer type.
    df.to_excel(iobytes, index=False)  # type: ignore[arg-type]
    excel_bytes = iobytes.getvalue()

    export_file_list.append(
        ExportFile(
            path=path,
            type=ExportFileType.EXCEL,
            bytes=excel_bytes,
            text=None,
        )
    )

### Utility: export_dataframe

In [ ]:
def export_dataframe(
    path: pathlib.Path,
    df: pd.DataFrame,
):
    export_dataframe_as_csv(path, df)
    export_dataframe_as_excel(path, df)

### Utility: export_file_bytes

In [ ]:
def export_file_bytes(
    path: pathlib.Path,
    file_bytes: bytes,
):
    export_file_list.append(
        ExportFile(
            path=path,
            type=ExportFileType.BYTES,
            bytes=file_bytes,
            text=None,
        )
    )

### Utility: export_markdown

In [ ]:
def export_markdown(
    path: pathlib.Path,
    markdown: str,
):
    path = path.with_suffix(path.suffix + ".md")

    export_file_list.append(
        ExportFile(
            path=path,
            type=ExportFileType.MARKDOWN,
            bytes=None,
            text=markdown,
        )
    )

### Utility: dataframe_sanitize

Sanitize contents of a dataframe that otherwise cannot be written to Excel.

In [ ]:
def dataframe_sanitize(df: pd.DataFrame) -> pd.DataFrame:
    def sanitize_cell(value):
        if isinstance(value, str):
            value = ILLEGAL_CHARACTERS_RE.sub("?", value).strip()

        return value

    return df.map(sanitize_cell)

### Utility: dataframe_format_export

Formats a dataframe for export.

In [ ]:
def dataframe_format_export(
    df: pd.DataFrame,
    *,
    drop_empty_columns: Optional[bool] = False,
    drop_columns: Optional[List[str]] = None,
    rename_columns: Optional[Dict[str, str]] = None,
    sort_columns: Optional[List[str]] = None,
    sort_rows_by_columns: Optional[List[str]] = None,
) -> pd.DataFrame:
    # Ensure we are modifying a copy.
    df = df.copy()

    # If requested, drop empty columns.
    if drop_empty_columns:
        empty_columns = []
        for column_current in df.columns:
            is_empty = bool(
                df[column_current].isnull().all()
                or (
                    df[column_current]
                    .astype(str)
                    .str.strip()
                    .isin(["", "nan", "None"])
                    .all()
                )
            )
            if is_empty:
                empty_columns.append(column_current)

        df = df.drop(columns=empty_columns)

    # If requested, drop specific columns.
    # Be robust to the possibility that a column is not present.
    if drop_columns:
        df = df.drop(columns=drop_columns, errors="ignore")

    # If requested, rename specific columns.
    if rename_columns:
        df = df.rename(columns=rename_columns)

    # Always sort "_clean" to the front.
    # If requested, sort other specific columns to the front.
    # Be robust to the possibility that a column is not present.
    # Be robust to the presence of additional columns.
    # Preserve existing order of additional columns after requested columns.
    sort_columns = ["_clean"] + (sort_columns or [])

    sort_columns = [
        column_current
        for column_current in sort_columns
        if column_current in df.columns
    ]
    sort_columns = sort_columns + [
        column_current
        for column_current in df.columns
        if column_current not in sort_columns
    ]

    df = df.loc[:, sort_columns]

    # If requested, sort rows by specific columns.
    # Be robust to the possibility that a column is not present.
    if sort_rows_by_columns:
        sort_rows_by_columns = [
            column_current
            for column_current in sort_rows_by_columns
            if column_current in df.columns
        ]

        df = df.sort_values(sort_rows_by_columns)

    return df

## Input

### Utility: archive_dir_path

In [ ]:
# Path containing encrypted archives.
archive_dir_path = "../../../secrets/data"

### Input Archive Suffix: archive_suffix

In [ ]:
# Obtain suffix indicating desired version of encrypted archives.
# Do not include the '.zip' suffix.
def input_archive_suffix():
    # Based on archives in our path, identify possible suffixes.
    pattern = re.compile("archive_(multicare|scca)_([^_]+)_(\\d{8})(?:_(.+))?.zip")
    archive_file_names = [
        name_current
        for name_current in os.listdir(archive_dir_path)
        if pattern.search(name_current)
    ]
    archive_matches = [
        re.match(pattern, name_current) for name_current in archive_file_names
    ]
    archive_suffixes = list(
        {
            "{}_{}{}".format(
                match_current.group(2),
                match_current.group(3),
                "_" + match_current.group(4)
                if (len(match_current.groups()) > 3 and match_current.group(4))
                else "",
            )
            for match_current in archive_matches
            if match_current is not None
        }
    )

    # If there is only one possible value, no need for a choice.
    archive_suffix = None
    if len(archive_suffixes) == 1:
        archive_suffix = archive_suffixes[0]
    else:
        for id_current, suffix_current in enumerate(archive_suffixes, 1):
            print("[{}]: {}".format(id_current, suffix_current))

        archive_suffix = archive_suffixes[
            int(input("Encrypted archive suffix index: ")) - 1
        ]

    print(archive_suffix)

    return archive_suffix


archive_suffix = input_archive_suffix()

### Input Archive Password: archive_password

In [ ]:
# Obtain password to encrypted archives.
archive_password = input("Encrypted archive password: ")

## Load Data

### Decrypt Archives

#### Data: archive_multicare

In [ ]:
def decrypt_archive_multicare():
    # Obtain name for each archive.
    archive_multicare_file_name = "archive_multicare_{}.zip".format(archive_suffix)

    # Obtain a full path to encrypted archive, relative to the location of the notebook.
    # Expects the encrypted archive to be in the "secrets/data" directory.
    archive_multicare_path = pathlib.Path(
        archive_dir_path,
        archive_multicare_file_name,
    )

    print("Decrypting archive:")
    print("{}".format(archive_multicare_path.resolve()))

    # Obtain the archive.
    archive_multicare = Archive.read_archive(
        archive_path=archive_multicare_path,
        password=archive_password,
    )

    print("{} documents.".format(len(archive_multicare.entries.values())))

    return archive_multicare


archive_multicare = decrypt_archive_multicare()

#### Data: archive_scca

In [ ]:
def decrypt_archive_scca():
    # Obtain name for each archive.
    archive_scca_file_name = "archive_scca_{}.zip".format(archive_suffix)

    # Obtain a full path to encrypted archive, relative to the location of the notebook.
    # Expects the encrypted archive to be in the "secrets/data" directory.
    archive_scca_path = pathlib.Path(
        archive_dir_path,
        archive_scca_file_name,
    )

    print("Decrypting archive:")
    print("{}".format(archive_scca_path.resolve()))

    # Obtain the archive.
    archive_scca = Archive.read_archive(
        archive_path=archive_scca_path,
        password=archive_password,
    )

    print("{} documents.".format(len(archive_scca.entries.values())))

    return archive_scca


archive_scca = decrypt_archive_scca()

### Decrypt MRN to RecordId

#### Data: mrn_to_record_id_bytes

#### Data: mrn_to_record_id

In [ ]:
def decrypt_mrn_to_record_id():
    # Obtain a full path to encrypted archive, relative to the location of the notebook.
    # Expects the encrypted archive to be in the "secrets/data" directory.
    archive_mrn_to_record_id_path = pathlib.Path(
        archive_dir_path,
        "archive_mrn_to_record_id.zip",
    )

    # Open the file
    with open(
        archive_mrn_to_record_id_path,
        mode="rb",
    ) as archive_file:
        with pyzipper.AESZipFile(
            archive_file,
            "r",
            compression=pyzipper.ZIP_LZMA,
            encryption=pyzipper.WZ_AES,
        ) as archive_zipfile:
            # Set the zipfile password
            archive_zipfile.setpassword(archive_password.encode("utf-8"))

            # Confirm the zipfile is valid
            if archive_zipfile.testzip():
                raise ValueError("Invalid archive or password")

            # Retrieve the Excel file.
            excel_bytes = archive_zipfile.read(
                "archive_mrn_to_record_id/archive_mrn_to_record_id.xlsx"
            )
            df_mrn_to_record_id = pd.read_excel(io.BytesIO(excel_bytes))

            # Ensure MRN are treated as strings.
            df_mrn_to_record_id["MRN"] = df_mrn_to_record_id["MRN"].astype(str)

            # Return the Excel file and a dictionary.
            return (
                excel_bytes,
                df_mrn_to_record_id.set_index("MRN")["recordId"].to_dict(),
            )


mrn_to_record_id_bytes, mrn_to_record_id = decrypt_mrn_to_record_id()

### Decrypt Cleaning

Decrypt one encrypted decision archive and load configured Excel tabs as DataFrames.

In [ ]:
@dataclasses.dataclass(frozen=True)
class CleaningArchiveEntry:
    """Excel path inside `archive_cleaning.zip`, worksheet name, and result field name."""

    name: str
    file: str
    tab: str


@dataclasses.dataclass
class CleaningArchiveData:
    """Named worksheets loaded from `archive_cleaning.zip` (see `cleaning_entries_to_load`)."""

    gad7_no_revision: pd.DataFrame
    gad7_with_revision: pd.DataFrame
    phq9_no_revision: pd.DataFrame
    phq9_with_revision: pd.DataFrame


cleaning_archive_path = pathlib.Path(
    archive_dir_path,
    "archive_cleaning.zip",
)

# Each entry points to an Excel file and tab (sheet) inside archive_cleaning.zip.
# `name` must match a field on `CleaningArchiveData`.
cleaning_entries_to_load: List[CleaningArchiveEntry] = [
    CleaningArchiveEntry(
        name="gad7_no_revision",
        file="archive_cleaning/GAD7 Cleaning-2026-05-01.xlsx",
        tab="GAD7 - No Revision",
    ),
    CleaningArchiveEntry(
        name="gad7_with_revision",
        file="archive_cleaning/GAD7 Cleaning-2026-05-01.xlsx",
        tab="GAD7 - with Revision for JAMES",
    ),
    CleaningArchiveEntry(
        name="phq9_no_revision",
        file="archive_cleaning/PHQ9 Cleaning-2026-04-22.xlsx",
        tab="PHQ9 - No Revisions",
    ),
    CleaningArchiveEntry(
        name="phq9_with_revision",
        file="archive_cleaning/PHQ9 Cleaning-2026-04-22.xlsx",
        tab="PHQ9 - Yes Revisions",
    ),
]


def decrypt_cleaning(
    *,
    entries_to_load: List[CleaningArchiveEntry],
) -> CleaningArchiveData:
    assert len(entries_to_load) > 0
    assert cleaning_archive_path.exists()

    expected_names = {
        field_current.name for field_current in dataclasses.fields(CleaningArchiveData)
    }
    entry_names = {entry_current.name for entry_current in entries_to_load}
    if entry_names != expected_names:
        raise ValueError(
            "cleaning_entries_to_load names {} must exactly match CleaningArchiveData fields {}".format(
                sorted(entry_names),
                sorted(expected_names),
            )
        )

    loaded_by_name: Dict[str, pd.DataFrame] = {}
    with open(cleaning_archive_path, mode="rb") as archive_file:
        with pyzipper.AESZipFile(
            archive_file,
            "r",
            compression=pyzipper.ZIP_LZMA,
            encryption=pyzipper.WZ_AES,
        ) as archive_zipfile:
            archive_zipfile.setpassword(archive_password.encode("utf-8"))

            archive_entry_names = set(archive_zipfile.namelist())
            for entry_current in entries_to_load:
                entry_file = entry_current.file
                entry_tab = entry_current.tab
                if entry_file not in archive_entry_names:
                    raise ValueError(
                        "Cleaning archive is missing file entry: {}".format(entry_file)
                    )

                entry_bytes = archive_zipfile.read(entry_file)
                with pd.ExcelFile(
                    io.BytesIO(entry_bytes),
                    engine="openpyxl",
                ) as excel_workbook:
                    sheet_names = excel_workbook.sheet_names
                    if entry_tab not in sheet_names:
                        raise ValueError(
                            "Cleaning worksheet {!r} not found in {!r}. "
                            "Available sheets (exact names): {!r}".format(
                                entry_tab,
                                entry_file,
                                sheet_names,
                            )
                        )
                    parsed_tab = excel_workbook.parse(
                        sheet_name=entry_tab,
                        dtype=str,
                    )
                if not isinstance(parsed_tab, pd.DataFrame):
                    raise ValueError(
                        "Expected one worksheet for sheet_name={!r}, got {!r}".format(
                            entry_tab,
                            type(parsed_tab).__name__,
                        )
                    )
                df_current = parsed_tab.copy()
                df_current["_cleanSource"] = "{}::{}".format(entry_file, entry_tab)
                loaded_by_name[entry_current.name] = df_current

    return CleaningArchiveData(**loaded_by_name)

In [ ]:
cleaning_archive_tables = decrypt_cleaning(
    entries_to_load=cleaning_entries_to_load,
)

## Process Archives

### Combine Patients DataFrames

In [ ]:
def combine_patients_dataframes():
    # Get patient documents from MultiCare.
    documents_multicare_patients = (
        archive_multicare.collection_documents(
            collection="patients",
        )
        .remove_sentinel()
        .remove_revisions()
    )
    df_multicare_patients = pd.DataFrame.from_records(
        documents_multicare_patients.documents
    )
    df_multicare_patients["database"] = "multicare"

    # Get patient documents from SCCA.
    documents_scca_patients = (
        archive_scca.collection_documents(
            collection="patients",
        )
        .remove_sentinel()
        .remove_revisions()
    )
    df_scca_patients = pd.DataFrame.from_records(documents_scca_patients.documents)
    df_scca_patients["database"] = "fhcc"

    # Unify all current patient documents.
    df_combined_patients = pd.concat(
        [df_multicare_patients, df_scca_patients]
    ).reset_index(drop=True)

    # Sanitize once so contents dataframe can be exported.
    df_combined_patients = dataframe_sanitize(df_combined_patients)

    return df_combined_patients


df_archive_patients_raw = combine_patients_dataframes()

### Reset Filtering and Sampling

#### Data: df_archive_patients

In [ ]:
df_archive_patients = df_archive_patients_raw.copy()

### Filter Pilot Patients

Remove the 6 pilot patients.

In [ ]:
df_archive_patients = df_archive_patients.drop(
    index=df_archive_patients[
        df_archive_patients["patientId"].isin(
            [
                "ymzwx6e6w6kqi",
                "mmmb54v52l7re",
                "ouoa4ucldbhie",
                "zazst4yu23a5q",
                "wf4btxqjtd2oa",
                "s3bcmgmp7gdss",
            ]
        )
    ].index.tolist()
).reset_index(drop=True)

### Sample Patients

In development, it can be helpful to sample a subset of patients.

In [ ]:
if DEVELOPMENT_SAMPLE_PATIENTS > 0:
    df_archive_patients = df_archive_patients.sample(
        n=DEVELOPMENT_SAMPLE_PATIENTS
    ).reset_index(drop=True)

### Utility: patient_documents

In [ ]:
# Create a helper for accessing the document collection of a unified patient.
def patient_documents(row_patient) -> document_set.DocumentSet:
    if row_patient["database"] == "multicare":
        archive = archive_multicare
    elif row_patient["database"] == "fhcc":
        archive = archive_scca
    else:
        raise ValueError()

    return archive.collection_documents(collection=row_patient["collection"])

## Prepare Cleaning

### Utility: build_cleaning_index

Each worksheet from `CleaningArchiveData` contributes `CleaningEntry` values.
Entries are keyed by `doc_id` (raw `_id`). Duplicate keys across tables raise.

In [ ]:
@dataclasses.dataclass(frozen=True)
class CleaningColumnEntry:
    name: str
    value: str


@dataclasses.dataclass(frozen=True)
class CleaningEntry:
    """One cleaning decision; `doc_id` matches raw document `_id` / export `_docId`."""

    doc_id: str
    delete: bool = False
    reviewed: bool = False
    clean_columns: Optional[List[CleaningColumnEntry]] = None

    def __post_init__(self) -> None:
        if not bson.objectid.ObjectId.is_valid(str(self.doc_id)):
            raise ValueError("CleaningEntry.doc_id is not a valid ObjectId")
        if self.delete and self.reviewed:
            raise ValueError(
                "CleaningEntry cannot be both delete=True and reviewed=True"
            )

    @property
    def clean_status(self) -> str:
        clean_parts: List[str] = []
        if self.delete:
            clean_parts.append("_delete")
        elif self.reviewed:
            clean_parts.append("_reviewed")
        if self.clean_columns is not None:
            clean_parts.extend(
                "_cleaned[{}]".format(column_current.name)
                for column_current in self.clean_columns
            )

        return ":".join(clean_parts)


def _normalize_clean_value(cell_clean: object, expected_clean_values: List[str]) -> str:
    cell_scalar = cast(Any, cell_clean)
    if pd.isna(cell_scalar):
        text_raw = ""
    else:
        text_raw = str(cell_scalar)

    normalized_current = " ".join(text_raw.split()).lower()

    expected_values = frozenset(expected_clean_values)

    assert normalized_current in expected_values, (
        "Unexpected _clean value: {!r}".format(cell_clean)
    )

    return normalized_current


def _clean_date_entry_error(
    date_cell: object,
    *,
    doc_id: object,
    clean_column_name: str,
) -> Optional[CleaningColumnEntry]:
    cell_scalar = cast(Any, date_cell)
    if pd.isna(cell_scalar):
        date_normalized = ""
    else:
        date_normalized = str(cell_scalar).strip()

    cleaned_date_value: Optional[datetime.date] = None

    fix_table = {
        # Exact `date_normalized` -> cleaned date; extend as needed.
        "0202-04-14": datetime.date(2022, 4, 14),
        "9202-08-18 00:00:00": datetime.date(2022, 8, 18),
        "9202-08-20 00:00:00": datetime.date(2022, 8, 20),
    }

    if date_normalized in fix_table:
        cleaned_date_value = fix_table[date_normalized]
    else:
        date_for_iso = date_normalized
        midnight_time_suffix = " 00:00:00"
        if date_for_iso.endswith(midnight_time_suffix):
            date_for_iso = date_for_iso[: -len(midnight_time_suffix)]

        try:
            parsed_date_value = datetime.date.fromisoformat(date_for_iso)
        except ValueError as exc:
            raise ValueError(
                "Unexpected date value for _docId={!r}: {!r}".format(doc_id, date_cell)
            ) from exc

        parsed_year = parsed_date_value.year

        if parsed_year < 100:
            cleaned_date_value = datetime.date(
                year=2000 + parsed_year,
                month=parsed_date_value.month,
                day=parsed_date_value.day,
            )
        elif 2000 <= parsed_year <= 2026:
            cleaned_date_value = None
        else:
            raise ValueError(
                "Unexpected date value for _docId={!r}: {!r}".format(doc_id, date_cell),
            )

    if cleaned_date_value is None:
        return None

    return CleaningColumnEntry(
        name=clean_column_name,
        value=cleaned_date_value.isoformat(),
    )


def print_cleaning_entries_summary(
    cleaning_table_key: str,
    cleaning_entries: Sequence[CleaningEntry],
    *,
    patched_column_name: str,
) -> None:
    entry_count = len(cleaning_entries)
    delete_entry_count = sum(1 for entry_current in cleaning_entries if entry_current.delete)
    reviewed_entry_count = sum(1 for entry_current in cleaning_entries if entry_current.reviewed)
    patched_column_fix_count = sum(
        1
        for entry_current in cleaning_entries
        if entry_current.clean_columns is not None
        and any(
            column_current.name == patched_column_name
            for column_current in entry_current.clean_columns
        )
    )

    print(
        "{cleaning_table_key} cleaning summary:\n"
        "  entries={entry_count}\n"
        "  delete={delete_entry_count}\n"
        "  reviewed={reviewed_entry_count}\n"
        "  {patched_column_name} column fixes={patched_column_fix_count}".format(
            cleaning_table_key=cleaning_table_key,
            entry_count=entry_count,
            delete_entry_count=delete_entry_count,
            reviewed_entry_count=reviewed_entry_count,
            patched_column_name=patched_column_name,
            patched_column_fix_count=patched_column_fix_count,
        )
    )


def cleaning_entries_for_gad7_no_revision(
    cleaning_table_key: str,
    _df: pd.DataFrame,
) -> List[CleaningEntry]:
    if "_docId" not in _df.columns:
        raise ValueError("Expected _docId column")
    if "_clean" not in _df.columns:
        raise ValueError("Expected _clean column")
    if "recordedDate" not in _df.columns:
        raise ValueError("Expected recordedDate column")

    cleaning_entries: List[CleaningEntry] = []

    for row_current in _df.to_dict(orient="records"):
        _normalize_clean_value(row_current["_clean"], ["", "fix date"])

        clean_column = _clean_date_entry_error(
            row_current["recordedDate"],
            doc_id=row_current["_docId"],
            clean_column_name="assessmentLogRecordedDate",
        )

        cleaning_entries.append(
            CleaningEntry(
                doc_id=str(row_current["_docId"]),
                reviewed=True,  # This file did not include explicit "keep" values.
                clean_columns=[clean_column] if clean_column is not None else None,
            )
        )

    print_cleaning_entries_summary(
        cleaning_table_key,
        cleaning_entries,
        patched_column_name="assessmentLogRecordedDate",
    )

    return cleaning_entries


def cleaning_entries_for_gad7_with_revision(
    cleaning_table_key: str,
    _df: pd.DataFrame,
) -> List[CleaningEntry]:
    if "_docId" not in _df.columns:
        raise ValueError("Expected _docId column")
    if "_clean" not in _df.columns:
        raise ValueError("Expected _clean column")
    if "recordedDate" not in _df.columns:
        raise ValueError("Expected recordedDate column")

    cleaning_entries: List[CleaningEntry] = []

    for row_current in _df.to_dict(orient="records"):
        clean_normalized = _normalize_clean_value(
            row_current["_clean"],
            ["", "delete", "keep"],
        )

        clean_column = _clean_date_entry_error(
            row_current["recordedDate"],
            doc_id=row_current["_docId"],
            clean_column_name="assessmentLogRecordedDate",
        )
        clean_columns_current = [clean_column] if clean_column is not None else None

        cleaning_entries.append(
            CleaningEntry(
                doc_id=str(row_current["_docId"]),
                delete=(clean_normalized == "delete"),
                reviewed=(clean_normalized == "keep"),
                clean_columns=clean_columns_current,
            )
        )

    print_cleaning_entries_summary(
        cleaning_table_key,
        cleaning_entries,
        patched_column_name="assessmentLogRecordedDate",
    )

    return cleaning_entries


def cleaning_entries_for_phq9_no_revision(
    cleaning_table_key: str,
    _df: pd.DataFrame,
) -> List[CleaningEntry]:
    if "_docId" not in _df.columns:
        raise ValueError("Expected _docId column")
    if "_clean" not in _df.columns:
        raise ValueError("Expected _clean column")
    if "recordedDate" not in _df.columns:
        raise ValueError("Expected recordedDate column")

    cleaning_entries: List[CleaningEntry] = []

    for row_current in _df.to_dict(orient="records"):
        clean_normalized = _normalize_clean_value(
            row_current["_clean"],
            [
                "",
                "date format",
                "date fornat",
                "delete - keep one on revisions with itemized scores",
            ],
        )

        clean_column = _clean_date_entry_error(
            row_current["recordedDate"],
            doc_id=row_current["_docId"],
            clean_column_name="assessmentLogRecordedDate",
        )
        clean_columns_current = [clean_column] if clean_column is not None else None

        row_delete = clean_normalized.startswith("delete")
        cleaning_entries.append(
            CleaningEntry(
                doc_id=str(row_current["_docId"]),
                delete=row_delete,
                reviewed=not row_delete,  # This file did not include explicit "keep" values.
                clean_columns=clean_columns_current,
            )
        )

    print_cleaning_entries_summary(
        cleaning_table_key,
        cleaning_entries,
        patched_column_name="assessmentLogRecordedDate",
    )

    return cleaning_entries


def cleaning_entries_for_phq9_with_revision(
    cleaning_table_key: str,
    _df: pd.DataFrame,
) -> List[CleaningEntry]:
    if "_docId" not in _df.columns:
        raise ValueError("Expected _docId column")
    if "_clean" not in _df.columns:
        raise ValueError("Expected _clean column")
    if "recordedDate" not in _df.columns:
        raise ValueError("Expected recordedDate column")

    cleaning_entries: List[CleaningEntry] = []

    for row_current in _df.to_dict(orient="records"):
        clean_normalized = _normalize_clean_value(
            row_current["_clean"],
            ["", "delete", "keep"],
        )

        clean_column = _clean_date_entry_error(
            row_current["recordedDate"],
            doc_id=row_current["_docId"],
            clean_column_name="assessmentLogRecordedDate",
        )
        clean_columns_current = [clean_column] if clean_column is not None else None

        cleaning_entries.append(
            CleaningEntry(
                doc_id=str(row_current["_docId"]),
                delete=(clean_normalized == "delete"),
                reviewed=(clean_normalized == "keep"),
                clean_columns=clean_columns_current,
            )
        )

    print_cleaning_entries_summary(
        cleaning_table_key,
        cleaning_entries,
        patched_column_name="assessmentLogRecordedDate",
    )

    return cleaning_entries


CLEANING_TABLE_ENTRY_BUILDERS: Dict[
    str,
    Callable[[str, pd.DataFrame], List[CleaningEntry]],
] = {
    "gad7_no_revision": cleaning_entries_for_gad7_no_revision,
    "gad7_with_revision": cleaning_entries_for_gad7_with_revision,
    "phq9_no_revision": cleaning_entries_for_phq9_no_revision,
    "phq9_with_revision": cleaning_entries_for_phq9_with_revision,
}


def build_cleaning_index(
    cleaning_tables: CleaningArchiveData,
) -> Dict[str, CleaningEntry]:
    cleaning_index_by_doc_id: Dict[str, CleaningEntry] = {}

    expected_table_names = {
        field_current.name for field_current in dataclasses.fields(CleaningArchiveData)
    }
    builder_names = set(CLEANING_TABLE_ENTRY_BUILDERS.keys())
    if expected_table_names != builder_names:
        raise ValueError(
            "CleaningArchiveData fields {} must match CLEANING_TABLE_ENTRY_BUILDERS keys {}".format(
                sorted(expected_table_names),
                sorted(builder_names),
            )
        )

    for field_current in dataclasses.fields(CleaningArchiveData):
        table_attribute = field_current.name
        builder_current = CLEANING_TABLE_ENTRY_BUILDERS[table_attribute]
        df_loaded = getattr(cleaning_tables, table_attribute)

        for entry_current in builder_current(table_attribute, df_loaded):
            doc_key = str(entry_current.doc_id).strip()
            if doc_key == "":
                raise ValueError(
                    "CleaningEntry has blank doc_id from table {}".format(table_attribute)
                )

            if doc_key in cleaning_index_by_doc_id:
                raise ValueError(
                    "Duplicate CleaningEntry for doc_id {!r}".format(doc_key),
                )

            if doc_key != entry_current.doc_id:
                entry_current = dataclasses.replace(entry_current, doc_id=doc_key)

            cleaning_index_by_doc_id[doc_key] = entry_current

    print("Built cleaning index with {} entries.".format(len(cleaning_index_by_doc_id)))

    return cleaning_index_by_doc_id

### Utility: apply_cleaning_index

In [ ]:
def apply_cleaning_index(
    documents: document_set.DocumentSet,
    cleaning_index: Dict[str, CleaningEntry],
) -> document_set.DocumentSet:
    cleaned_documents = []
    for document_current in documents.documents:
        document_cleaned = dict(document_current)
        document_id_key = str(document_current["_id"])
        cleaning_entry = cleaning_index.get(document_id_key)
        clean_status = ""

        if cleaning_entry is not None:
            clean_status = cleaning_entry.clean_status
            if cleaning_entry.clean_columns is not None:
                for clean_column_current in cleaning_entry.clean_columns:
                    document_cleaned[clean_column_current.name] = clean_column_current.value

        document_cleaned["_clean"] = clean_status
        cleaned_documents.append(document_cleaned)

    return document_set.DocumentSet(
        documents=cleaned_documents
    )

### Data: cleaning_index

In [ ]:
cleaning_index = build_cleaning_index(cleaning_archive_tables)

## Prepare Data

### Prepare Patients

#### Documentation: Patients

All patients included in this export, based on `patientIdentity` documents.
These are stored separately from the collection of documents associated with each patient.
They provide an index of all patients, then we can access the collection of documents for each individual patient.

A `patientIdentity` document may have been modified throughout the study (e.g., to change a patient name or email address).
If the document was modified, this export includes only the final version.
There is therefore exactly one row per patient.

Data is originally taken from two database exports: from FHCC and from MultiCare.
Raw data are merged, then a `database` column is added to indicate the origin of each patient.

There were 6 pilot patients. These have been completely removed.

TODO: The patient with `recordId` `1747` maps to two different `MRN` and `patientId`.
This was because they moved from one system to another during the study.
There is still a need to decide how to special case that patient.

#### Data: df_patients_raw

In [ ]:
df_patients_raw = df_archive_patients.copy()

#### Data: df_patients

Remove most fields, so they are not needlessly visible during export script development.

Use `MRN` to map each patient to a `recordId`.

In [ ]:
df_patients = df_patients_raw.copy()

# Ensure MRN are treated as strings.
df_patients["MRN"] = df_patients["MRN"].astype(str)

# Apply a transform to look up the recordId.
def transform_add_record_id(
    df_patients: pd.DataFrame,
) -> pd.DataFrame:
    def _transform_add_record_id_from_mrn(row):
        return str(mrn_to_record_id.get(row["MRN"], "???"))

    df_patients = df_patients.copy()
    df_patients["recordId"] = df_patients.apply(
        _transform_add_record_id_from_mrn, axis=1
    )

    return df_patients


df_patients = transform_add_record_id(df_patients)

# "collection" cannot be removed at this point, as it is used by later processing.
df_patients = dataframe_format_export(
    df_patients,
    drop_columns=[
        # Remove clutter.
        "_id",
        "_rev",
        "_set_id",
        "_type",
        # Remove identifiers.
        "cognitoAccount",  # Includes email.
        "name",
    ],
    sort_columns=[
        "recordId",
        "database",
        "patientId",
        "MRN",
        "collection",
    ],
    sort_rows_by_columns=[
        "recordId",
        "database",
        "patientId",
    ],
)

#### Data: patient_id_to_record_id

In [ ]:
patient_id_to_record_id = (
    df_patients.copy().set_index("patientId")["recordId"].to_dict()
)

### Prepare Per-Patient DocumentSets and DataFrames

#### Data: patient_id_to_documentset

In [ ]:
def prepare_patient_id_to_documentset():
    patient_id_to_documentset = {}

    progress_max = len(df_patients)
    progress_patient_count = ipywidgets.IntProgress(min=0, max=progress_max)

    print("Preparing DocumentSet for {} Patients".format(progress_max))
    IPython.display.display(progress_patient_count)

    progress_patient_count.description = "{}/{}".format(0, progress_max)
    for patient_count, (row_current, patient_current) in enumerate(
        df_patients.iterrows()
    ):
        patient_id_current = patient_current["patientId"]
        patient_collection = patient_documents(patient_current.to_dict())
        patient_collection = apply_cleaning_index(
            patient_collection,
            cleaning_index,
        )

        patient_id_to_documentset[patient_id_current] = patient_collection

        progress_patient_count.description = "{}/{}".format(
            patient_count + 1, progress_max
        )
        progress_patient_count.value = patient_count + 1

    return patient_id_to_documentset


patient_id_to_documentset = prepare_patient_id_to_documentset()

#### Transform: transform_add_created

Add a `_created` column to each document.

In [ ]:
def transform_add_created(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    def _transform_add_created_from_id(row):
        datetime_parsed = bson.objectid.ObjectId(row["_id"]).generation_time.astimezone(
            pytz.utc
        )
        datetime_pacific = datetime_parsed.astimezone(
            pytz.timezone("America/Los_Angeles")
        )

        return datetime_pacific.strftime("%Y-%m-%dT%H:%M:%S")

    df_documents = df_documents.copy()
    df_documents["_created"] = df_documents.apply(
        _transform_add_created_from_id, axis=1
    )

    return df_documents

#### Transform: transform_add_record_id_and_patient_id

Add a `patientId` column to each document.

In [ ]:
def transform_add_record_id_and_patient_id(
    df_documents: pd.DataFrame,
    *,
    patient_id,
) -> pd.DataFrame:
    df_documents = df_documents.copy()
    df_documents["recordId"] = patient_id_to_record_id[patient_id]
    df_documents["_patientId"] = patient_id

    return df_documents

#### Data: patient_id_to_df_documents_raw

In [ ]:
def prepare_patient_id_to_df_documents_raw():
    patient_id_to_df_documents_raw = {}

    progress_max = len(patient_id_to_documentset)
    progress_patient_count = ipywidgets.IntProgress(min=0, max=progress_max)

    print("Preparing DataFrame for {} Patients".format(progress_max))
    IPython.display.display(progress_patient_count)

    progress_patient_count.description = "{}/{}".format(0, progress_max)
    for patient_count, (patient_id_current, patient_documentset_current) in enumerate(
        patient_id_to_documentset.items()
    ):
        df_documents_raw_current = pd.DataFrame.from_records(
            patient_documentset_current.documents
        )

        # Sanitize contents for export.
        df_documents_raw_current = dataframe_sanitize(df_documents_raw_current)

        # Apply minimal transforms to the "raw" documents.
        df_documents_raw_current = transform_add_created(
            df_documents_raw_current,
        )
        df_documents_raw_current = transform_add_record_id_and_patient_id(
            df_documents_raw_current, patient_id=patient_id_current
        )

        patient_id_to_df_documents_raw[patient_id_current] = df_documents_raw_current

        progress_patient_count.description = "{}/{}".format(
            patient_count + 1, progress_max
        )
        progress_patient_count.value = patient_count + 1

    return patient_id_to_df_documents_raw


patient_id_to_df_documents_raw = prepare_patient_id_to_df_documents_raw()

In [ ]:
df_documents_raw = pd.concat(patient_id_to_df_documents_raw.values(), ignore_index=True)

## Transform Documents

### Documentation: Common Fields

Several common fields are shared across different analysis exports:

- `_docType` identifies the type of the document.

- `recordId` identifies the patient, according to the study id.

- `_patientId` identifies the patient, according to the database id.

- `_docId` uniquely identifies a document within the collection for a `_patientId`.

  The combination of `_patientId` and `_docId` are persistent and unique.
  This can support inspection and data cleaning, but is unlikely to be direclty used in analyses.

- If there could be multiple instances of a document type (e.g., multiple assessment logs, multiple mood logs),
  then documents include a type-specific identifier (e.g., `_assessmentLogId`, `_moodLogId`).

- `_rev` identifies the version of the document (i.e., the revision).
  If only one instance of a document type is allowed (e.g., a single safety plan),
  this identifies revisions of that document over time.
  If there could be multiple instances of a document type,
  the combination of the type-specific identifier and the version identifies revisions of specific instances.

- `_created` indicates when a document was created, recovered from an encoding within `_docId`.
  This is provided in Pacific time.

- `_deleted` indicates that a revision deleted the document.

### Documentation: Activities

Exported from `activity` documents. A single row is included for each `activity`.

Includes common fields documented in `commonFields.md`.

Notable transformations:

- `valueLifeArea` and `valueName` are calculated via lookup of `valueId`.

### Transform: transform_activities

In [ ]:
def transform_activities(
    df_documents: pd.DataFrame,
    documents: document_set.DocumentSet,
) -> pd.DataFrame:
    # Deleted rows will not have a id, so restore that from the _set_id.
    def _transform_activity_id_deleted(row):
        if (
            row["_type"] == "activity"
            and pd.notna(row.get("_deleted", None))
            and row.get("_deleted", False)
        ):
            return row["_set_id"]

        return row.get("activityId", None)

    # Use the valueId to recover a valueLifeArea.
    def _transform_expand_value_life_area(row):
        if row["_type"] != "activity":
            return row.get("valueLifeArea", None)

        if pd.isna(row.get("valueId", None)) or not row.get("valueId", None):
            return None

        value_document = documents.filter_match(
            match_type="value",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"valueId": row["valueId"]},
        ).unique()

        return value_document["lifeAreaId"]

    # Use the valueId to recover a valueName.
    def _transform_expand_value_name(row):
        if row["_type"] != "activity":
            return row.get("valueName", None)

        if pd.isna(row.get("valueId", None)) or not row.get("valueId", None):
            return None

        value_document = documents.filter_match(
            match_type="value",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"valueId": row["valueId"]},
        ).unique()

        return value_document["name"]

    df_documents = df_documents.copy()
    df_documents["activityId"] = df_documents.apply(
        _transform_activity_id_deleted, axis=1
    )
    df_documents["valueLifeArea"] = df_documents.apply(
        _transform_expand_value_life_area, axis=1
    )
    df_documents["valueName"] = df_documents.apply(_transform_expand_value_name, axis=1)

    return df_documents

### Documentation: Activity Logs

Exported from `activityLog` documents. A single row is included for each `activityLog`.

Includes common fields documented in `commonFields.md`.

Notable values:

- All values of `_rev` are `1`, as it was not possible to edit an activity log.

- Documents include both `_created` and `dueDate`.

  - Per common fields, `_created` indicates when a document was created.

  - `dueDate` indicates what date the activity due in an `activitySchedule`.

Notable transformations:

- `value` and `activity` fields are calculated from a snapshot stored with the `activityLog`.

### Transform: transform_activity_logs

In [ ]:
def transform_activity_logs(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    def _transform_snapshot_value_id(row):
        if row["_type"] != "activityLog":
            return row.get("valueId", None)

        if "value" not in row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]:
            return None

        return row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["value"][
            "valueId"
        ]

    def _transform_snapshot_value_life_area(row):
        if row["_type"] != "activityLog":
            return row.get("valueLifeArea", None)

        if "value" not in row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]:
            return None

        return row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["value"][
            "lifeAreaId"
        ]

    def _transform_snapshot_value_name(row):
        if row["_type"] != "activityLog":
            return row.get("valueLifeName", None)

        if "value" not in row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]:
            return None

        return row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["value"]["name"]

    def _transform_snapshot_activity_id(row):
        if row["_type"] != "activityLog":
            return row.get("activityId", None)

        return row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["activity"][
            "activityId"
        ]

    def _transform_snapshot_activity_enjoyment(row):
        if row["_type"] != "activityLog":
            return row.get("activityEnjoyment", None)

        if (
            "enjoyment"
            not in row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["activity"]
        ):
            return None

        return row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["activity"][
            "enjoyment"
        ]

    def _transform_snapshot_activity_importance(row):
        if row["_type"] != "activityLog":
            return row.get("activityImportance", None)

        if (
            "importance"
            not in row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["activity"]
        ):
            return None

        return row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["activity"][
            "importance"
        ]

    def _transform_snapshot_activity_name(row):
        if row["_type"] != "activityLog":
            return row.get("activityName", None)

        return row["dataSnapshot"]["scheduledActivity"]["dataSnapshot"]["activity"][
            "name"
        ]

    # Format a dueDate.
    def _transform_snapshot_due_date(row):
        if row["_type"] != "activityLog":
            return row.get("dueDate", None)

        date_parsed = date_utils.parse_date(
            date=row["dataSnapshot"]["scheduledActivity"]["dueDate"]
        )

        return date_parsed.strftime("%Y-%m-%d")

    df_documents = df_documents.copy()
    df_documents["valueId"] = df_documents.apply(_transform_snapshot_value_id, axis=1)
    df_documents["valueLifeArea"] = df_documents.apply(
        _transform_snapshot_value_life_area, axis=1
    )
    df_documents["valueName"] = df_documents.apply(
        _transform_snapshot_value_name, axis=1
    )
    df_documents["dueDate"] = df_documents.apply(_transform_snapshot_due_date, axis=1)
    df_documents["activityId"] = df_documents.apply(
        _transform_snapshot_activity_id, axis=1
    )
    df_documents["activityEnjoyment"] = df_documents.apply(
        _transform_snapshot_activity_enjoyment, axis=1
    )
    df_documents["activityImportance"] = df_documents.apply(
        _transform_snapshot_activity_importance, axis=1
    )
    df_documents["activityName"] = df_documents.apply(
        _transform_snapshot_activity_name, axis=1
    )

    return df_documents

### Documentation: Activity Schedules

Exported from `activitySchedule` documents. A single row is included for each `activitySchedule`.

Includes common fields documented in `commonFields.md`.

- Documents include both `_created` and `scheduledDate`.

  - Per common fields, `_created` indicates when a document was created.

  - `scheduledDate` indicates what date was indicated for an `activitySchedule`.

Notable transformations:

- `activityEnjoyment`, `activityImportance`, `activityName`, `valueId` are calculated via lookup of `activityId`.

- `valueLifeArea` and `valueName` are calculated via lookup of `valueId`.

- `scheduleRepeatDays` is calculated from a raw `repeatDayFlags`.

### Transform: transform_activity_schedules

In [ ]:
def transform_activity_schedules(
    df_documents: pd.DataFrame,
    documents: document_set.DocumentSet,
) -> pd.DataFrame:
    # Deleted rows will not have a id, so restore that from the _set_id.
    def _transform_activity_schedule_id_deleted(row):
        if (
            row["_type"] == "activitySchedule"
            and pd.notna(row.get("_deleted", None))
            and row.get("_deleted", False)
        ):
            return row["_set_id"]

        return row.get("activityScheduleId", None)

    # Use the activityId to recover an enjoyment.
    def _transform_expand_activity_enjoyment(row):
        if row["_type"] != "activitySchedule":
            return row.get("activityEnjoyment", None)

        if pd.isna(row.get("activityId", None)) or not row.get("activityId", None):
            return None

        activity_document = documents.filter_match(
            match_type="activity",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"activityId": row["activityId"]},
        ).unique()

        return activity_document.get("enjoyment", None)

    # Use the activityId to recover an importance.
    def _transform_expand_activity_importance(row):
        if row["_type"] != "activitySchedule":
            return row.get("activityImportance", None)

        if pd.isna(row.get("activityId", None)) or not row.get("activityId", None):
            return None

        activity_document = documents.filter_match(
            match_type="activity",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"activityId": row["activityId"]},
        ).unique()

        return activity_document.get("importance", None)

    # Use the activityId to recover a name.
    def _transform_expand_activity_name(row):
        if row["_type"] != "activitySchedule":
            return row.get("activityName", None)

        if pd.isna(row.get("activityId", None)) or not row.get("activityId", None):
            return None

        activity_document = documents.filter_match(
            match_type="activity",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"activityId": row["activityId"]},
        ).unique()

        return activity_document["name"]

    # Use the activityId to recover a valueId.
    def _transform_expand_activity_value_id(row):
        if row["_type"] != "activitySchedule":
            return row.get("valueId", None)

        if pd.isna(row.get("activityId", None)) or not row.get("activityId", None):
            return None

        activity_document = documents.filter_match(
            match_type="activity",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"activityId": row["activityId"]},
        ).unique()

        return activity_document.get("valueId", None)

    # Use the valueId to recover a valueLifeArea.
    def _transform_expand_value_life_area(row):
        if row["_type"] != "activitySchedule":
            return row.get("valueLifeArea", None)

        if pd.isna(row.get("valueId", None)) or not row.get("valueId", None):
            return None

        value_document = documents.filter_match(
            match_type="value",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"valueId": row["valueId"]},
        ).unique()

        return value_document["lifeAreaId"]

    # Use the valueId to recover a valueName.
    def _transform_expand_value_name(row):
        if row["_type"] != "activitySchedule":
            return row.get("valueName", None)

        if pd.isna(row.get("valueId", None)) or not row.get("valueId", None):
            return None

        value_document = documents.filter_match(
            match_type="value",
            match_deleted=False,
            match_datetime_at=document_set.datetime_from_document_id(
                document_id=row["_id"]
            ),
            match_values={"valueId": row["valueId"]},
        ).unique()

        return value_document["name"]

    # Recover the repeat day flags.
    def _transform_repeat_day_flags(row):
        if row["_type"] != "activitySchedule":
            return row.get("repeatDayFlags", None)

        if pd.isna(row.get("repeatDayFlags", None)) or not row.get(
            "repeatDayFlags", None
        ):
            return None

        result = ""
        for day_of_week_current in scope.enums.DayOfWeek:
            if row["repeatDayFlags"][day_of_week_current.value]:
                result += day_of_week_current.value[0:2]

        return result

    df_documents = df_documents.copy()
    df_documents["activityScheduleId"] = df_documents.apply(
        _transform_activity_schedule_id_deleted, axis=1
    )
    df_documents["activityEnjoyment"] = df_documents.apply(
        _transform_expand_activity_enjoyment, axis=1
    )
    df_documents["activityImportance"] = df_documents.apply(
        _transform_expand_activity_importance, axis=1
    )
    df_documents["activityName"] = df_documents.apply(
        _transform_expand_activity_name, axis=1
    )
    df_documents["valueId"] = df_documents.apply(
        _transform_expand_activity_value_id, axis=1
    )
    df_documents["valueLifeArea"] = df_documents.apply(
        _transform_expand_value_life_area, axis=1
    )
    df_documents["valueName"] = df_documents.apply(_transform_expand_value_name, axis=1)
    df_documents["repeatDayFlags"] = df_documents.apply(
        _transform_repeat_day_flags, axis=1
    )

    return df_documents

### Documentation: Assessments

Exported from `assessment` documents. A single row is included for each `assessment`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_assessments

In [ ]:
def transform_assessments(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    df_documents = df_documents.copy()

    # Medication tracking was never activated.
    df_documents = df_documents.loc[
        (df_documents["_type"] != "assessment")
        | (
            (df_documents["_type"] == "assessment")
            & (df_documents["assessmentId"] != "medication")
        )
    ]

    return df_documents

### Documentation: Assessment Logs

Exported from `assessmentLog` documents. A single row is included for each `assessmentLog`.

Includes common fields documented in `commonFields.md`.

Notable values:

- `assessmentId` will be either `gad-7` or `phq-9`.

- Documents include both `_created` and `recordedDate`.

  - Per common fields, `_created` indicates when a document was created.

  - `recordedDate` indicates what date was indicated for an `assessmentLog`.

  - For assessment logs submitted via the patient app, these were the same.

  - For assessment logs submitted via the provider registry, the provider entered the date of the assessment log.

Notable transformations:

- `recordedDate` is calculated in Pacific time from a raw `recordedDateTime`.

- `submittedBy` is calculated from a raw `patientSubmitted`.

- If individual scale components were available in a raw `pointValues`,
  they were promoted to columns (e.g., `gad7Anxious`, `phq9Interest`).

- An assessment score (i.e., `gad7Score`, `phq9Score`)
  was taken from a raw `totalScore` or by summing the values in a raw `pointValues`.

TODO: Inspecting a sample of revised assessment logs suggests there will not be a simple approach to data cleaning.

TODO: `submittedBy` value of `SW` is potentially misleading.
These were created using the registry, but not necessarily by a social worker.

TODO: `todo_scheduledAssessmentId` is present for patient-submitted entries, may allow recovering information about an assessment schedule.
It is unclear how correct this field is, so it may be best to not rely upon it.

TODO: `todo_submittedByProviderId` is present for registry-submitted entries, may allow recovering information about who submitted an entry.

### Transform: transform_assessment_logs

In [ ]:
def transform_assessment_logs(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # Pull each value of the gad-7 assessment scale out to its own column.
    def _transform_gad7_points(
        df_documents: pd.DataFrame,
    ) -> pd.DataFrame:
        def _factory_transform_gad7_points_key(key_json):
            def _transform_gad7_points_key(row):
                if row["_type"] != "assessmentLog":
                    return None
                if row["assessmentId"] != "gad-7":
                    return None
                if not row["pointValues"]:
                    return None

                return row["pointValues"][key_json]

            return _transform_gad7_points_key

        gad7_enum_map = {
            "Anxious": "gad7Anxious",
            "Constant worrying": "gad7ConstantWorrying",
            "Worrying too much": "gad7WorryingTooMuch",
            "Trouble relaxing": "gad7TroubleRelaxing",
            "Restless": "gad7Restless",
            "Irritable": "gad7Irritable",
            "Afraid": "gad7Afraid",
        }

        for (key_json, key_export) in gad7_enum_map.items():
            df_documents[key_export] = df_documents.apply(
                _factory_transform_gad7_points_key(key_json), axis=1
            )

        return df_documents

    # Obtain or calculate a gad-7 score.
    def _transform_gad7_score(row):
        if row["_type"] != "assessmentLog":
            return row.get("gad7Score", None)
        if row["assessmentId"] != "gad-7":
            return row.get("gad7Score", None)

        # Some rows already provide a totalScore.
        if "totalScore" in row and not pd.isna(row["totalScore"]):
            return row["totalScore"]

        # Otherwise we need to sum the pointValues.
        if "pointValues" in row and row["pointValues"]:
            return sum(row["pointValues"].values())

        # We should always have one or the other.
        raise ValueError()

    # Pull each value of the phq-9 assessment scale out to its own column.
    def _transform_phq9_points(
        df_documents: pd.DataFrame,
    ) -> pd.DataFrame:
        def _factory_transform_phq9_points_key(key_json):
            def _transform_phq9_points_key(row):
                if row["_type"] != "assessmentLog":
                    return None
                if row["assessmentId"] != "phq-9":
                    return None
                if not row["pointValues"]:
                    return None

                return row["pointValues"][key_json]

            return _transform_phq9_points_key

        phq9_enum_map = {
            "Interest": "phq9Interest",
            "Mood": "phq9Mood",
            "Sleep": "phq9Sleep",
            "Energy": "phq9Energy",
            "Appetite": "phq9Appetite",
            "Guilt": "phq9Guilt",
            "Concentrating": "phq9Concentrating",
            "Motor": "phq9Motor",
            "Suicide": "phq9Suicide",
        }

        for (key_json, key_export) in phq9_enum_map.items():
            df_documents[key_export] = df_documents.apply(
                _factory_transform_phq9_points_key(key_json), axis=1
            )

        return df_documents

    # Obtain or calculate a phq-9 score.
    def _transform_phq9_score(row):
        if row["_type"] != "assessmentLog":
            return row.get("phq9Score", None)
        if row["assessmentId"] != "phq-9":
            return row.get("phq9Score", None)

        # Some rows already provide a totalScore.
        if "totalScore" in row and not pd.isna(row["totalScore"]):
            return row["totalScore"]

        # Otherwise we need to sum the pointValues.
        if "pointValues" in row and row["pointValues"]:
            return sum(row["pointValues"].values())

        # We should always have one or the other.
        raise ValueError()

    # Format a recordedDate.
    def _transform_recorded_date(row):
        if row["_type"] != "assessmentLog":
            return row.get("assessmentLogRecordedDate", None)

        # Respect any explicit value already present (e.g., cleaning overrides).
        recorded_date_existing = row.get("assessmentLogRecordedDate", None)
        if recorded_date_existing is not None and not pd.isna(recorded_date_existing):
            return str(recorded_date_existing)

        datetime_parsed = date_utils.parse_datetime(datetime=row["recordedDateTime"])
        datetime_pacific = datetime_parsed.astimezone(
            pytz.timezone("America/Los_Angeles")
        )

        return datetime_pacific.strftime("%Y-%m-%d")

    def _transform_scheduled_assessment_id(row):
        if row["_type"] != "assessmentLog":
            return row.get("assessmentLogScheduledAssessmentId", None)

        # Logs labeled on-demand should all be submitted by a social worker.
        if row["scheduledAssessmentId"] == "on-demand":
            if row["patientSubmitted"] == True:
                raise ValueError()

            return None

        return row["scheduledAssessmentId"]

    # Format a submittedBy column as requested.
    def _transform_submitted_by(row):
        if row["_type"] != "assessmentLog":
            return row.get("assessmentLogSubmittedBy", None)

        if row["patientSubmitted"] == True:
            return "Pt"
        elif row["patientSubmitted"] == False:
            return "SW"
        else:
            raise ValueError()

    df_documents = df_documents.copy()
    df_documents = _transform_gad7_points(df_documents)
    df_documents["gad7Score"] = df_documents.apply(_transform_gad7_score, axis=1)
    df_documents = _transform_phq9_points(df_documents)
    df_documents["phq9Score"] = df_documents.apply(_transform_phq9_score, axis=1)
    df_documents["assessmentLogRecordedDate"] = df_documents.apply(
        _transform_recorded_date, axis=1
    )
    df_documents["assessmentLogScheduledAssessmentId"] = df_documents.apply(
        _transform_scheduled_assessment_id, axis=1
    )
    df_documents["assessmentLogSubmittedBy"] = df_documents.apply(
        _transform_submitted_by, axis=1
    )

    return df_documents

### Documentation: Case Reviews

Exported from `caseReview` documents. A single row is included for each `caseReview`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_case_reviews

In [ ]:
def transform_case_reviews(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # Elevate up the name of the psychiatrist.
    def _transform_consulting_psychiatrist(row):
        if row["_type"] != "caseReview":
            return row.get("consultingPsychiatrist", None)

        return row["consultingPsychiatrist"]["name"]

    # Format date of case review.
    def _transform_case_review_date(row):
        if row["_type"] != "caseReview":
            return row.get("date", None)

        date_parsed = date_utils.parse_date(
            date=row["date"]
        )

        return date_parsed.strftime("%Y-%m-%d")

    df_documents = df_documents.copy()
    df_documents["consultingPsychiatrist"] = df_documents.apply(_transform_consulting_psychiatrist, axis=1)
    df_documents["date"] = df_documents.apply(_transform_case_review_date, axis=1)

    return df_documents

### Documentation: Clinical History

Exported from `clinicalHistory` documents. A single row is included for each `clinicalHistory`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_clinical_histories

In [ ]:
def transform_clinical_histories(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # Expand currentTreatmentRegimen (cancerTreatmentRegimenFlags) into one column per flag.
    def _factory_transform_current_treatment_regimen_key(key_json):
        def _transform_key(row):
            if row["_type"] != "clinicalHistory":
                return None
            regimen = row.get("currentTreatmentRegimen")
            if not isinstance(regimen, dict):
                return None
            return regimen.get(key_json)

        return _transform_key

    current_treatment_regimen_enum_map = {
        "Surgery": "currentTreatmentRegimenSurgery",
        "Chemotherapy": "currentTreatmentRegimenChemotherapy",
        "Radiation": "currentTreatmentRegimenRadiation",
        "Stem Cell Transplant": "currentTreatmentRegimenStemCellTransplant",
        "Immunotherapy": "currentTreatmentRegimenImmunotherapy",
        "CAR-T": "currentTreatmentRegimenCART",
        "Endocrine": "currentTreatmentRegimenEndocrine",
        "Surveillance": "currentTreatmentRegimenSurveillance",
        "Other": "currentTreatmentRegimenOtherFlag",
    }

    df_documents = df_documents.copy()
    for (key_json, key_export) in current_treatment_regimen_enum_map.items():
        df_documents[key_export] = df_documents.apply(
            _factory_transform_current_treatment_regimen_key(key_json), axis=1
        )

    return df_documents

### Documentation: Mood Logs

Exported from `moodLog` documents. A single row is included for each `moodLog`.

Includes common fields documented in `commonFields.md`.

Notable values:

- All values of `_rev` are `1`, as it was not possible to edit a mood log.

### Transform: transform_mood_logs

In [ ]:
def transform_mood_logs(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # No transformations are needed.
    return df_documents

### Documentation: Patient Profile

Exported from `profile` (patientProfile) documents. A single row is included for each `profile`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_patient_profiles

In [ ]:
def transform_patient_profiles(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # Flatten primaryCareManager to primaryCareManagerName.
    def _transform_primary_care_manager_name(row):
        if row["_type"] != "profile":
            return None
        primary_care_manager = row.get("primaryCareManager")
        if not isinstance(primary_care_manager, dict):
            return None
        return primary_care_manager.get("name")

    # Expand race (PatientRaceFlags) into one column per flag.
    def _factory_transform_race_key(key_json):
        def _transform_key(row):
            if row["_type"] != "profile":
                return None
            race_obj = row.get("race")
            if not isinstance(race_obj, dict):
                return None
            return race_obj.get(key_json)

        return _transform_key

    race_enum_map = {
        "American Indian or Alaska Native": "raceAmericanIndianOrAlaskaNative",
        "Asian or Asian American": "raceAsianOrAsianAmerican",
        "Black or African American": "raceBlackOrAfricanAmerican",
        "Native Hawaiian or Other Pacific Islander": "raceNativeHawaiianOrOtherPacificIslander",
        "White": "raceWhite",
        "Unknown": "raceUnknown",
    }

    # Expand discussionFlag (DiscussionFlags) into one column per flag.
    def _factory_transform_discussion_flag_key(key_json):
        def _transform_key(row):
            if row["_type"] != "profile":
                return None
            discussion_flag = row.get("discussionFlag")
            if not isinstance(discussion_flag, dict):
                return None
            return discussion_flag.get(key_json)

        return _transform_key

    discussion_flag_enum_map = {
        "Flag as safety risk": "discussionFlagFlagAsSafetyRisk",
        "Flag for discussion": "discussionFlagFlagForDiscussion",
    }

    # Format enrollmentDate as YYYY-MM-DD.
    def _transform_enrollment_date(row):
        if row["_type"] != "profile":
            return row.get("enrollmentDate", None)
        if pd.isna(row.get("enrollmentDate", None)) or not row.get("enrollmentDate"):
            return None
        date_parsed = date_utils.parse_date(date=row["enrollmentDate"])
        return date_parsed.strftime("%Y-%m-%d")

    df_documents = df_documents.copy()
    df_documents["enrollmentDate"] = df_documents.apply(
        _transform_enrollment_date, axis=1
    )
    df_documents["primaryCareManagerName"] = df_documents.apply(
        _transform_primary_care_manager_name, axis=1
    )
    for (key_json, key_export) in race_enum_map.items():
        df_documents[key_export] = df_documents.apply(
            _factory_transform_race_key(key_json), axis=1
        )
    for (key_json, key_export) in discussion_flag_enum_map.items():
        df_documents[key_export] = df_documents.apply(
            _factory_transform_discussion_flag_key(key_json), axis=1
        )

    return df_documents

### Documentation: Review Mark

Exported from `reviewMark` documents. A single row is included for each `reviewMark`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_review_marks

In [ ]:
def transform_review_marks(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # No transformations are needed.
    return df_documents

### Documentation: Safety Plan

Exported from `safetyPlan` documents. A single row is included for each `safetyPlan`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_safety_plans

In [ ]:
def transform_safety_plans(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # No transformations are needed.
    return df_documents

### Documentation: Scheduled Assessment

Exported from `scheduledAssessment` documents. A single row is included for each `scheduledAssessment`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_scheduled_assessments

In [ ]:
def transform_scheduled_assessments(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # No transformations are needed.
    return df_documents

### Documentation: Scheduled Activity

Exported from `scheduledActivity` documents. A single row is included for each `scheduledActivity`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_scheduled_activities

In [ ]:
def transform_scheduled_activities(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # No transformations are needed.
    return df_documents

### Documentation: Session

Exported from `session` documents. A single row is included for each `session`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_sessions

In [ ]:
def transform_sessions(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # Expand behavioralStrategyChecklist (behavioralStrategyChecklistFlags) into one column per flag.
    def _factory_transform_behavioral_strategy_checklist_key(key_json):
        def _transform_key(row):
            if row["_type"] != "session":
                return None
            checklist = row.get("behavioralStrategyChecklist")
            if not isinstance(checklist, dict):
                return None
            return checklist.get(key_json)

        return _transform_key

    behavioral_strategy_checklist_enum_map = {
        "Behavioral Activation": "behavioralStrategyChecklistBehavioralActivation",
        "Motivational Interviewing": "behavioralStrategyChecklistMotivationalInterviewing",
        "Problem Solving Therapy": "behavioralStrategyChecklistProblemSolvingTherapy",
        "Cognitive Therapy": "behavioralStrategyChecklistCognitiveTherapy",
        "Mindfulness Strategies": "behavioralStrategyChecklistMindfulnessStrategies",
        "Supportive Therapy": "behavioralStrategyChecklistSupportiveTherapy",
        "Other": "behavioralStrategyChecklistOther",
    }

    # Expand behavioralActivationChecklist (bAChecklistFlags) into one column per flag.
    def _factory_transform_behavioral_activation_checklist_key(key_json):
        def _transform_key(row):
            if row["_type"] != "session":
                return None
            checklist = row.get("behavioralActivationChecklist")
            if not isinstance(checklist, dict):
                return None
            return checklist.get(key_json)

        return _transform_key

    behavioral_activation_checklist_enum_map = {
        "Review of the BA model": "behavioralActivationChecklistReviewOfTheBAModel",
        "Values and goals assessment": "behavioralActivationChecklistValuesAndGoalsAssessment",
        "Activity scheduling": "behavioralActivationChecklistActivityScheduling",
        "Mood and activity monitoring": "behavioralActivationChecklistMoodAndActivityMonitoring",
        "Relaxation": "behavioralActivationChecklistRelaxation",
        "Positive reinforcement": "behavioralActivationChecklistPositiveReinforcement",
        "Managing avoidance behaviors": "behavioralActivationChecklistManagingAvoidanceBehaviors",
        "Problem-solving": "behavioralActivationChecklistProblemSolving",
    }

    df_documents = df_documents.copy()
    for (key_json, key_export) in behavioral_strategy_checklist_enum_map.items():
        df_documents[key_export] = df_documents.apply(
            _factory_transform_behavioral_strategy_checklist_key(key_json), axis=1
        )
    for (key_json, key_export) in behavioral_activation_checklist_enum_map.items():
        df_documents[key_export] = df_documents.apply(
            _factory_transform_behavioral_activation_checklist_key(key_json), axis=1
        )

    return df_documents

### Documentation: Values

Exported from `value` documents. A single row is included for each `value`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_values

In [ ]:
def transform_values(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # Deleted rows will not have a id, so restore that from the _set_id.
    def _transform_value_id_deleted(row):
        if (
            row["_type"] == "value"
            and pd.notna(row.get("_deleted", None))
            and row.get("_deleted", False)
        ):
            return row["_set_id"]

        return row.get("valueId", None)

    df_documents = df_documents.copy()
    df_documents["valueId"] = df_documents.apply(_transform_value_id_deleted, axis=1)

    return df_documents

### Documentation: Values Inventory

Exported from `valuesInventory` documents. A single row is included for each `valuesInventory`.

Includes common fields documented in `commonFields.md`.

### Transform: transform_values_inventories

In [ ]:
def transform_values_inventories(
    df_documents: pd.DataFrame,
) -> pd.DataFrame:
    # No transformations are needed.
    return df_documents

### Utility: apply_transforms

In [ ]:
def apply_transforms(
    df_documents: pd.DataFrame,
    documents: document_set.DocumentSet,
) -> pd.DataFrame:
    df_documents = df_documents.copy()
    df_documents = transform_activities(
        df_documents,
        documents,
    )
    df_documents = transform_activity_logs(
        df_documents,
    )
    df_documents = transform_activity_schedules(
        df_documents,
        documents,
    )
    df_documents = transform_assessments(
        df_documents,
    )
    df_documents = transform_assessment_logs(
        df_documents,
    )
    df_documents = transform_case_reviews(
        df_documents,
    )
    df_documents = transform_clinical_histories(
        df_documents,
    )
    df_documents = transform_mood_logs(
        df_documents,
    )
    df_documents = transform_patient_profiles(
        df_documents,
    )
    df_documents = transform_review_marks(
        df_documents,
    )
    df_documents = transform_safety_plans(
        df_documents,
    )
    df_documents = transform_scheduled_assessments(
        df_documents,
    )
    df_documents = transform_scheduled_activities(
        df_documents,
    )
    df_documents = transform_sessions(
        df_documents,
    )
    df_documents = transform_values(
        df_documents,
    )
    df_documents = transform_values_inventories(
        df_documents,
    )
    return df_documents

### Transform: timeline_documents_from_documentset

In [ ]:
def timeline_documents_from_documentset(
    patient_id: str,
    document_set: document_set.DocumentSet,
) -> pd.DataFrame:
    """
    Build a dataframe of timeline event rows from the DocumentSet only.
    Caller merges the result with the existing documents dataframe.
    """

    record_id = patient_id_to_record_id.get(patient_id, "")

    def _calculate_event_study_enrollment() -> Dict[str, object]:
        """Find final profile, recover enrollment date. Raises if missing."""
        profile = document_set.filter_match(
            match_type="profile",
            match_deleted=False,
        ).remove_revisions().unique()

        enrollment_date = profile.get("enrollmentDate")
        if not enrollment_date:
            raise ValueError("Profile has no enrollmentDate")
        enrollment_date = date_utils.parse_date(date=enrollment_date).strftime("%Y-%m-%d")

        return {
            "_docType": "timelineEvent",
            "recordId": record_id,
            "_patientId": patient_id,
            "_timelineDate": enrollment_date,
            "_timelineEvent": "studyEnrollment",
        }

    def _calculate_events_assessment_log_by_patient() -> List[Dict[str, object]]:
        assessment_logs = document_set.filter_match(
            match_type="assessmentLog",
            match_deleted=False,
            match_values={"patientSubmitted": True},
        )

        events = []
        for assessment_log in assessment_logs.documents:
            # Assessment logs are required to have a date.
            log_date = str(assessment_log["recordedDateTime"])
            log_date = date_utils.parse_datetime(datetime=log_date).strftime("%Y-%m-%d")

            events.append({
                "_docType": "timelineEvent",
                "recordId": record_id,
                "_patientId": patient_id,
                "_timelineDate": log_date,
                "_timelineEvent": "assessmentLogByPatient",
                "assessmentId": assessment_log["assessmentId"],
            })

        return events

    def _calculate_events_mood_log_by_patient() -> List[Dict[str, object]]:
        mood_logs = document_set.filter_match(
            match_type="moodLog",
            match_deleted=False,
        )
        events = []
        for log in mood_logs.documents:
            log_date = str(log["recordedDateTime"])
            log_date = date_utils.parse_datetime(datetime=log_date).strftime("%Y-%m-%d")

            events.append({
                "_docType": "timelineEvent",
                "recordId": record_id,
                "_patientId": patient_id,
                "_timelineDate": log_date,
                "_timelineEvent": "moodLogByPatient",
            })

        return events

    # def _calculate_event_study_end() -> Optional[Dict[str, object]]:
    #     """Find first profile with status End; return timeline event dict or None."""
    #     status_end = scope.enums.DepressionTreatmentStatus.End.value
    #     profiles_with_end = document_set.filter_match(
    #         match_type="profile",
    #         match_deleted=False,
    #         match_values={"depressionTreatmentStatus": status_end},
    #     )
    #     if not profiles_with_end:
    #         return None
    #     docs = profiles_with_end.documents
    #     first_by_time = min(
    #         docs,
    #         key=lambda doc: datetime_from_document(document=doc),
    #     )
    #     print(json.dumps(first_by_time, indent=2))
    #     end_date = datetime_from_document(
    #         document=first_by_time
    #     ).strftime("%Y-%m-%d")
    #     return {
    #         "_docType": "timelineEvent",
    #         "recordId": record_id,
    #         "_patientId": patient_id,
    #         "_timelineEvent": "studyEnd",
    #         "_timelineDate": end_date,
    #     }

    # def _calculate_event_study_end_scheduled(event_study_enrollment: Dict[str, object]) -> Dict[str, object]:
    #     enrollment_date = str(event_study_enrollment["_timelineDate"])
    #     enrollment_date = datetime.date.fromisoformat(enrollment_date)
    #     end_scheduled_date = (
    #         enrollment_date + datetime.timedelta(days=365)
    #     ).strftime("%Y-%m-%d")

    #     return {
    #         "_docType": "timelineEvent",
    #         "recordId": record_id,
    #         "_patientId": patient_id,
    #         "_timelineEvent": "studyEndScheduled",
    #         "_timelineDate": end_scheduled_date,
    #     }

    event_study_enrollment = _calculate_event_study_enrollment()
    # event_study_end = _calculate_event_study_end()
    # event_study_end_scheduled = (
    #     _calculate_event_study_end_scheduled(event_study_enrollment)
    #     if event_study_end is None
    #     else None
    # )

    events_assessment_log_by_patient = _calculate_events_assessment_log_by_patient()
    events_mood_log_by_patient = _calculate_events_mood_log_by_patient()

    # Collect events that may be single dicts, lists of dicts, or None.
    events_raw = [
        event_study_enrollment,
        # event_study_end,
        # event_study_end_scheduled,
        events_assessment_log_by_patient,
        events_mood_log_by_patient,
    ]

    events_normalized: List[Dict[str, object]] = []
    for event in events_raw:
        if event is None:
            continue
        if isinstance(event, list):
            events_normalized.extend(event)
        else:
            events_normalized.append(event)

    return pd.DataFrame(events_normalized)

### Data: patient_id_to_df_documents

In [ ]:
def prepare_transform_patient_documents():
    patient_id_to_df_documents = {}

    progress_max = len(patient_id_to_df_documents_raw)
    progress_patient_count = ipywidgets.IntProgress(min=0, max=progress_max)

    print("Transforming DataFrame for {} Patients".format(progress_max))
    IPython.display.display(progress_patient_count)

    progress_patient_count.description = "{}/{}".format(0, progress_max)
    for patient_count, (patient_id_current, df_documents_raw_current) in enumerate(
        patient_id_to_df_documents_raw.items()
    ):
        df_documents_current = apply_transforms(
            df_documents_raw_current.copy(),
            patient_id_to_documentset[patient_id_current],
        )
        df_documents_timeline = timeline_documents_from_documentset(
            patient_id_current,
            patient_id_to_documentset[patient_id_current],
        )
        patient_id_to_df_documents[patient_id_current] = pd.concat(
            [df_documents_current, df_documents_timeline],
            ignore_index=True,
        )

        progress_patient_count.description = "{}/{}".format(
            patient_count + 1, progress_max
        )
        progress_patient_count.value = patient_count + 1

    return patient_id_to_df_documents


patient_id_to_df_documents = prepare_transform_patient_documents()

### Prepare Combined Documents DataFrame

#### Data: df_documents_raw

In [ ]:
df_documents_raw = pd.concat(patient_id_to_df_documents_raw.values(), ignore_index=True)

#### Data: df_documents

In [ ]:
df_documents = pd.concat(patient_id_to_df_documents.values(), ignore_index=True)

## Export

### Reset Export File List

In [ ]:
export_file_list.clear()

### Documentation: Export

This folder contains an export of SCOPE platform data.
This includes patient data, should always be stored in an encrypted format, and should be treated as highly sensitive.
Tables are commonly exported in both Excel and comma-separated formats
(i.e., `.xlsx` via `pd.DataFrame.to_excel`, `.csv` via `pd.DataFrame.to_csv`).
Excel will typically be easier to visually inspect,
while comma-separated may be easier for R-based analyses.

The root folder contains exports that are intended to be used in analyses.

- `patients` is a list of all patients included in the export. Documentation in `patients.md`.
- `activities` is an export of all `activity` documents. Documentation in `activities.md`.
- `activityLogs` is an export of all `activityLog` documents. Documentation in `activityLogs.md`.
- `activitySchedules` is an export of all `activitySchedule` documents. Documentation in `activitySchedules.md`.
- `assessments` is an export of all `assessment` documents. Documentation in `assessments.md`.
- `assessmentsGad7` is an export of all GAD-7 `assessmentLog` documents. Documentation in `assessmentLogs.md`.
- `assessmentsPhq9` is an export of all PHQ-9 `assessmentLog` documents. Documentation in `assessmentLogs.md`.
- `clinicalHistory` is an export of all `clinicalHistory` documents. Documentation in `clinicalHistory.md`.
- `moodLogs` is an export of all `moodLog` documents. Documentation in `moodLogs.md`.
- `patientProfiles` is an export of all `profile` documents. Documentation in `patientProfiles.md`.
- `reviewMarks` is an export of all `reviewMark` documents. Documentation in `reviewMarks.md`.
- `safetyPlans` is an export of all `safetyPlan` documents. Documentation in `safetyPlans.md`.
- `scheduledAssessments` is an export of all `scheduledAssessment` documents. Documentation in `scheduledAssessments.md`.
- `scheduledActivities` is an export of all `scheduledActivity` documents. Documentation in `scheduledActivities.md`.
- `sessions` is an export of all `session` documents. Documentation in `sessions.md`.
- `values` is an export of all `value` documents. Documentation in `values.md`.
- `valuesInventories` is an export of all `valuesInventory` documents. Documentation in `valuesInventories.md`.

Several of the above data types are related.

- A person may have configured one or more `value`.
- A person may have configured one or more `activity`. Each may have a `value` associated with it.
- A `activity` may have one or more `activitySchedule` configured. These may be one-time or repeating.
- Creation of an `activitySchedule` also created a set of `scheduledActivity` instances (i.e., one for each day the activity was scheduled).
- A person could complete an `activityLog` corresponding to a `scheduledActivity`.

There was also a significant redesign of the relationship between `value` and `activity` documents in the `v0.7.0` release deployed on 2023-04-29.
- Release notes at: https://github.com/uwscope/scope-web/releases/tag/v0.7.0
- Prior to that release, a person was required to first create a `value` and then an `activity`. Every `activity` had an associated `value`.
- Beginning with that release, a person could create a `value` or an `activity` independently. Associating an `activity` with a `value` became optional.

The `data` folder contains additional exports, intended to support inspection of and communication around underlying data.
Exports are prepared in a multi-step process, converting a database of JSON documents into tables for analyses.

- Note that document tables are sparse, as most documents (i.e., rows) do not contain most values (i.e., columns).
  Each row includes a `_type` taken from the JSON document, commonly used for filtering to specific types of documents.

- Raw JSON documents are first loaded into a table for each patient.
  This is the raw underlying data, included in the export to allow inspection.
  The `data/patients` folder includes a folder for each patient, named according to the `patientId`.

- Tables for all patients are combined in a single large table, exported as `documents.raw`.
  This is all of the raw data and contains everything that is available.

- A series of transformations are applied to the single large table.
  Each transformation computes one or more new columns that are added to the single large table.

- After all transformations are completed, the single large table is exported as `documents.transformed`.
  Filtering the single large table by `patientId`, transformed tables are also exported to the each folder in `data/patients`.
  These are included in the export to allow inspection.

- Tables intended for analyses are then exported as different subsets of `documents.transformed`.

The `config` folder contains additional configuration that was used in the export.
If a change in configuration is needed, these files should be modified and then used in a new export.

- `archive_mrn_to_record_id.xlsx` contains the mapping from `MRN` to `recordId` that was used in this export.


In [ ]:
export_markdown(pathlib.Path("documentation"), documentation_as_markdown("Export"))
export_markdown(
    pathlib.Path("commonFields"), documentation_as_markdown("Common Fields")
)

### Patients

- Documented above in "Documentation: Patients Export".

In [ ]:
export_markdown(
    pathlib.Path("patients"),
    documentation_as_markdown("Patients"),
)

export_file_bytes(
    pathlib.Path(
        "config",
        "archive_mrn_to_record_id.xlsx",
    ),
    mrn_to_record_id_bytes,
)

export_dataframe(
    pathlib.Path(
        "data",
        "patients.raw",
    ),
    df_patients_raw,
)

export_dataframe(
    pathlib.Path(
        "patients",
    ),
    dataframe_format_export(
        df_patients,
        drop_columns=["collection"],
    ),
)

### Per-Patient Documents

In [ ]:
def export_per_patient_documents():
    progress_max = len(df_patients)
    progress_patient_count = ipywidgets.IntProgress(min=0, max=progress_max)

    print("Per-Patient Export for {} Patients".format(progress_max))
    IPython.display.display(progress_patient_count)

    progress_patient_count.description = "{}/{}".format(0, progress_max)
    for patient_count, (row_current, patient_current) in enumerate(
        df_patients.iterrows()
    ):
        patient_id_current = patient_current["patientId"]

        df_documents_raw_current = patient_id_to_df_documents_raw[patient_id_current]
        export_dataframe(
            pathlib.Path(
                "data",
                "patients",
                "patient_{}".format(patient_id_current),
                "patient_{}.raw".format(patient_id_current),
            ),
            df_documents_raw_current,
        )

        df_documents_current = patient_id_to_df_documents[patient_id_current]
        export_dataframe(
            pathlib.Path(
                "data",
                "patients",
                "patient_{}".format(patient_id_current),
                "patient_{}.transformed".format(patient_id_current),
            ),
            df_documents_current,
        )

        progress_patient_count.description = "{}/{}".format(
            patient_count + 1, progress_max
        )
        progress_patient_count.value = patient_count + 1


if DEVELOPMENT_EXPORT_PER_PATIENT_DOCUMENTS:
    export_per_patient_documents()

### Combined Documents

In [ ]:
if DEVELOPMENT_EXPORT_COMBINED_DOCUMENTS:
    export_dataframe(
        pathlib.Path(
            "data",
            "documents.raw",
        ),
        df_documents_raw,
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "documents.transformed",
        ),
        df_documents,
    )

### Analysis: Activities

In [ ]:
def export_analysis_activities():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("activities"),
        documentation_as_markdown("Activities"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "activities.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[df_documents_raw["_type"] == "activity"],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "activities.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "activity"],
            drop_empty_columns=True,
        ),
    )

    # Formatted values.
    drop_columns = [
        "_set_id",
        "editedDateTime",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
        "activityId": "_activityId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "_activityId",
        "_rev",
        "_created",
        "_deleted",
        "valueId",
        "valueLifeArea",
        "valueName",
        "enjoyment",
        "importance",
        "name",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("activities"),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "activity"],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Activity Logs

In [ ]:
def export_analysis_activity_logs():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("activityLogs"),
        documentation_as_markdown("Activity Logs"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "activityLogs.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[df_documents_raw["_type"] == "activityLog"],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "activityLogs.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "activityLog"],
            drop_empty_columns=True,
        ),
    )

    # Formatted values.
    drop_columns = [
        "_set_id",
        "recordedDateTime",
        "dataSnapshot",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
        "activityLogId": "_activityLogId",
        "success": "completed",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "_activityLogId",
        "_rev",
        "_created",
        "scheduledActivityId",
        "valueId",
        "valueLifeArea",
        "valueName",
        "activityId",
        "activityEnjoyment",
        "activityImportance",
        "activityName",
        "dueDate",
        "completed",
        "accomplishment",
        "pleasure",
        "alternative",
        "comment",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("activityLogs"),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "activityLog"],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Activity Schedules

In [ ]:
def export_analysis_activity_schedules():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("activitySchedules"),
        documentation_as_markdown("Activity Schedules"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "activitySchedules.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[df_documents_raw["_type"] == "activitySchedule"],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "activitySchedules.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "activitySchedule"],
            drop_empty_columns=True,
        ),
    )

    # Formatted values.
    drop_columns = [
        "_set_id",
        "editedDateTime",
        "hasReminder",  # Junk reminder field that was never implemented.
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
        "activityScheduleId": "_activityScheduleId",
        "date": "scheduleDate",
        "timeOfDay": "scheduleTimeOfDay",
        "hasRepetition": "scheduleRepeat",
        "repeatDayFlags": "scheduleRepeatDays",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "_activityScheduleId",
        "_rev",
        "_created",
        "_deleted",
        "valueId",
        "valueLifeArea",
        "valueName",
        "activityId",
        "activityEnjoyment",
        "activityImportance",
        "activityName",
        "scheduleDate",
        "scheduleTimeOfDay",
        "scheduleRepeat",
        "scheduleRepeatDays",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("activitySchedules"),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "activitySchedule"],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Assessments

In [ ]:
def export_analysis_assessments():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("assessments"),
        documentation_as_markdown("Assessments"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "assessments.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "assessment"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "assessments.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "assessment"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted assessment documents.
    drop_columns = [
        "_set_id",
        "assignedDateTime",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "assessmentId",
        "_rev",
        "_created",
        "assigned",
        "dayOfWeek",
        "frequency",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "assessmentId",
        "_rev",
    ]

    export_dataframe(
        pathlib.Path("assessments"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "assessment"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Assessment Logs

In [ ]:
def export_analysis_assessment_logs():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("assessmentLogs"),
        documentation_as_markdown("Assessment Logs"),
    )

    # Preliminary GAD-7 documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "assessmentLogsGad7.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                (df_documents_raw["_type"] == "assessmentLog")
                & (df_documents_raw["assessmentId"] == "gad-7")
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "assessmentLogsGad7.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                (df_documents["_type"] == "assessmentLog")
                & (df_documents["assessmentId"] == "gad-7")
            ],
            drop_empty_columns=True,
        ),
    )

    # Preliminary PHQ-9 documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "assessmentLogsPhq9.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                (df_documents_raw["_type"] == "assessmentLog")
                & (df_documents_raw["assessmentId"] == "phq-9")
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "assessmentLogsPhq9.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                (df_documents["_type"] == "assessmentLog")
                & (df_documents["assessmentId"] == "phq-9")
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted GAD-7 and PHQ-9 documents.
    drop_columns = [
        "_type",
        "_set_id",
        "patientSubmitted",
        "pointValues",
        "recordedDateTime",
        "scheduledAssessmentId",
        "totalScore",
    ]
    rename_columns = {
        "_id": "_docId",
        "assessmentLogId": "_assessmentLogId",
        "assessmentLogRecordedDate": "recordedDate",
        "assessmentLogScheduledAssessmentId": "todo_scheduledAssessmentId",
        "submittedByProviderId": "todo_submittedByProviderId",
        "assessmentLogSubmittedBy": "submittedBy",
    }
    sort_columns = [
        "recordId",
        "_patientId",
        "_docId",
        "_assessmentLogId",
        "_rev",
        "_created",
        "assessmentId",
        "recordedDate",
        "submittedBy",
        "todo_scheduledAssessmentId",
        "todo_submittedByProviderId",
        "gad7Anxious",
        "gad7ConstantWorrying",
        "gad7WorryingTooMuch",
        "gad7TroubleRelaxing",
        "gad7Restless",
        "gad7Irritable",
        "gad7Afraid",
        "gad7Score",
        "phq9Interest",
        "phq9Mood",
        "phq9Sleep",
        "phq9Energy",
        "phq9Appetite",
        "phq9Guilt",
        "phq9Concentrating",
        "phq9Motor",
        "phq9Suicide",
        "phq9Score",
        "comment",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_assessmentLogId",
        "_rev",
    ]

    df_gad7 = df_documents.loc[
        (df_documents["_type"] == "assessmentLog")
        & (df_documents["assessmentId"] == "gad-7")
    ]
    gad7_id_revision_count = df_gad7.groupby("assessmentLogId").size()

    export_dataframe(
        pathlib.Path("assessmentLogsGad7"),
        dataframe_format_export(
            df_gad7,
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

    # GAD-7 assessment logs with only a single revision (never edited).
    single_revision_gad7_ids = gad7_id_revision_count[gad7_id_revision_count == 1].index
    df_gad7_single_revision = df_gad7.loc[df_gad7["assessmentLogId"].isin(single_revision_gad7_ids)]

    export_dataframe(
        pathlib.Path("assessmentLogsGad7NoRevision"),
        dataframe_format_export(
            df_gad7_single_revision,
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

    # GAD-7 assessment logs with at least one revision (edited at least once).
    with_revision_gad7_ids = gad7_id_revision_count[gad7_id_revision_count > 1].index
    df_gad7_with_revision = df_gad7.loc[df_gad7["assessmentLogId"].isin(with_revision_gad7_ids)]

    export_dataframe(
        pathlib.Path("assessmentLogsGad7WithRevision"),
        dataframe_format_export(
            df_gad7_with_revision,
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

    df_phq9 = df_documents.loc[
        (df_documents["_type"] == "assessmentLog")
        & (df_documents["assessmentId"] == "phq-9")
    ]
    phq9_id_revision_count = df_phq9.groupby("assessmentLogId").size()

    export_dataframe(
        pathlib.Path("assessmentLogsPhq9"),
        dataframe_format_export(
            df_phq9,
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

    # PHQ-9 assessment logs with only a single revision (never edited).
    single_revision_phq9_ids = phq9_id_revision_count[phq9_id_revision_count == 1].index
    df_phq9_single_revision = df_phq9.loc[df_phq9["assessmentLogId"].isin(single_revision_phq9_ids)]

    export_dataframe(
        pathlib.Path("assessmentLogsPhq9NoRevision"),
        dataframe_format_export(
            df_phq9_single_revision,
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

    # PHQ-9 assessment logs with at least one revision (edited at least once).
    with_revision_phq9_ids = phq9_id_revision_count[phq9_id_revision_count > 1].index
    df_phq9_with_revision = df_phq9.loc[df_phq9["assessmentLogId"].isin(with_revision_phq9_ids)]

    export_dataframe(
        pathlib.Path("assessmentLogsPhq9WithRevision"),
        dataframe_format_export(
            df_phq9_with_revision,
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Case Reviews

In [ ]:
def export_analysis_case_reviews():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("caseReviews"),
        documentation_as_markdown("Case Reviews"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "caseReviews.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "caseReview"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "caseReviews.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "caseReview"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted caseReview documents.
    drop_columns = [
        "_set_id",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
        "date": "caseReviewDate",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "caseReviewId",
        "_rev",
        "_created",
        "caseReviewDate",
        "consultingPsychiatrist",
        "behavioralStrategyChange",
        "medicationChange",
        "otherRecommendations",
        "referralsChange",
        "reviewNote",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "caseReviewDate",
        "caseReviewId",
        "_rev",
    ]

    export_dataframe(
        pathlib.Path("caseReviews"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "caseReview"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Clinical History

In [ ]:
def export_analysis_clinical_histories():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("clinicalHistory"),
        documentation_as_markdown("Clinical History"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "clinicalHistory.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "clinicalHistory"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "clinicalHistory.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "clinicalHistory"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted clinical history documents.
    drop_columns = [
        "_set_id",
        "currentTreatmentRegimen",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "_rev",
        "_created",
        "primaryCancerDiagnosis",
        "dateOfCancerDiagnosis",
        "currentTreatmentRegimenSurgery",
        "currentTreatmentRegimenChemotherapy",
        "currentTreatmentRegimenRadiation",
        "currentTreatmentRegimenStemCellTransplant",
        "currentTreatmentRegimenImmunotherapy",
        "currentTreatmentRegimenCART",
        "currentTreatmentRegimenEndocrine",
        "currentTreatmentRegimenSurveillance",
        "currentTreatmentRegimenOtherFlag",
        "currentTreatmentRegimenOther",
        "currentTreatmentRegimenNotes",
        "psychDiagnosis",
        "pastPsychHistory",
        "pastSubstanceUse",
        "psychSocialBackground",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_rev",
    ]

    export_dataframe(
        pathlib.Path("clinicalHistory"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "clinicalHistory"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Mood Logs

In [ ]:
def export_analysis_mood_logs():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("moodLogs"),
        documentation_as_markdown("Mood Logs"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "moodLogs.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[df_documents_raw["_type"] == "moodLog"],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "moodLogs.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "moodLog"],
            drop_empty_columns=True,
        ),
    )

    # Formatted mood logs.
    drop_columns = [
        "_type",
        "_set_id",
        "recordedDateTime",
    ]
    rename_columns = {
        "_id": "_docId",
        "moodLogId": "_moodLogId",
    }
    sort_columns = [
        "recordId",
        "_patientId",
        "_docId",
        "_moodLogId",
        "_rev",
        "_created",
        "mood",
        "comment",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("moodLogs"),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "moodLog"],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Patient Profile

In [ ]:
def export_analysis_patient_profiles():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("patientProfiles"),
        documentation_as_markdown("Patient Profile"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "patientProfiles.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "profile"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "patientProfiles.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "profile"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted patient profile documents.
    drop_columns = [
        "_set_id",
        "primaryCareManager",
        "race",
        "discussionFlag",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "_rev",
        "_created",
        "name",
        "MRN",
        "clinicCode",
        "birthdate",
        "sex",
        "gender",
        "pronoun",
        "raceAmericanIndianOrAlaskaNative",
        "raceAsianOrAsianAmerican",
        "raceBlackOrAfricanAmerican",
        "raceNativeHawaiianOrOtherPacificIslander",
        "raceWhite",
        "raceUnknown",
        "ethnicity",
        "primaryOncologyProvider",
        "primaryCareManagerName",
        "discussionFlagFlagAsSafetyRisk",
        "discussionFlagFlagForDiscussion",
        "followupSchedule",
        "depressionTreatmentStatus",
        "site",
        "enrollmentDate",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_rev",
    ]

    export_dataframe(
        pathlib.Path("patientProfiles"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "profile"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Review Mark

In [ ]:
def export_analysis_review_marks():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("reviewMarks"),
        documentation_as_markdown("Review Mark"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "reviewMarks.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "reviewMark"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "reviewMarks.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "reviewMark"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted review mark documents.
    drop_columns = [
        "_set_id",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "reviewMarkId",
        "_rev",
        "_created",
        "editedDateTime",
        "effectiveDateTime",
        "providerId",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("reviewMarks"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "reviewMark"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Safety Plan

In [ ]:
def export_analysis_safety_plans():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("safetyPlans"),
        documentation_as_markdown("Safety Plan"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "safetyPlans.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "safetyPlan"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "safetyPlans.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "safetyPlan"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted safety plan documents.
    drop_columns = [
        "_set_id",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "_rev",
        "_created",
        "assigned",
        "assignedDateTime",
        "lastUpdatedDateTime",
        "reasonsForLiving",
        "warningSigns",
        "copingStrategies",
        "socialDistractions",
        "settingDistractions",
        "supporters",
        "professionals",
        "urgentServices",
        "safeEnvironment",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_rev",
    ]

    export_dataframe(
        pathlib.Path("safetyPlans"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "safetyPlan"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Scheduled Assessment

In [ ]:
def export_analysis_scheduled_assessments():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("scheduledAssessments"),
        documentation_as_markdown("Scheduled Assessment"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "scheduledAssessments.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "scheduledAssessment"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "scheduledAssessments.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "scheduledAssessment"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted scheduled assessment documents.
    drop_columns = [
        "_set_id",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "scheduledAssessmentId",
        "_rev",
        "_created",
        "_deleted",
        "assessmentId",
        "dueDate",
        "dueTimeOfDay",
        "dueDateTime",
        "reminderDate",
        "reminderTimeOfDay",
        "reminderDateTime",
        "completed",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("scheduledAssessments"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "scheduledAssessment"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Scheduled Activity

In [ ]:
def export_analysis_scheduled_activities():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("scheduledActivities"),
        documentation_as_markdown("Scheduled Activity"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "scheduledActivities.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "scheduledActivity"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "scheduledActivities.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "scheduledActivity"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted scheduled activity documents.
    drop_columns = [
        "_set_id",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "scheduledActivityId",
        "_rev",
        "_created",
        "activityScheduleId",
        "dataSnapshot",
        "dueDate",
        "dueTimeOfDay",
        "dueDateTime",
        "completed",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("scheduledActivities"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "scheduledActivity"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Session

In [ ]:
def export_analysis_sessions():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("sessions"),
        documentation_as_markdown("Session"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "sessions.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "session"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "sessions.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "session"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted session documents.
    drop_columns = [
        "_set_id",
        "behavioralStrategyChecklist",
        "behavioralActivationChecklist",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "sessionId",
        "_rev",
        "_created",
        "date",
        "sessionType",
        "billableMinutes",
        "medicationChange",
        "currentMedications",
        "behavioralStrategyChecklistBehavioralActivation",
        "behavioralStrategyChecklistMotivationalInterviewing",
        "behavioralStrategyChecklistProblemSolvingTherapy",
        "behavioralStrategyChecklistCognitiveTherapy",
        "behavioralStrategyChecklistMindfulnessStrategies",
        "behavioralStrategyChecklistSupportiveTherapy",
        "behavioralStrategyChecklistOther",
        "behavioralStrategyOther",
        "behavioralActivationChecklistReviewOfTheBAModel",
        "behavioralActivationChecklistValuesAndGoalsAssessment",
        "behavioralActivationChecklistActivityScheduling",
        "behavioralActivationChecklistMoodAndActivityMonitoring",
        "behavioralActivationChecklistRelaxation",
        "behavioralActivationChecklistPositiveReinforcement",
        "behavioralActivationChecklistManagingAvoidanceBehaviors",
        "behavioralActivationChecklistProblemSolving",
        "referrals",
        "otherRecommendations",
        "sessionNote",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("sessions"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "session"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Values

In [ ]:
def export_analysis_values():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("values"),
        documentation_as_markdown("Values"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "values.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[df_documents_raw["_type"] == "value"],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "values.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "value"],
            drop_empty_columns=True,
        ),
    )

    # Formatted values.
    drop_columns = [
        "_type",
        "_set_id",
        "editedDateTime",
    ]
    rename_columns = {
        "_id": "_docId",
        "valueId": "_valueId",
        "lifeAreaId": "lifeArea",
    }
    sort_columns = [
        "recordId",
        "_patientId",
        "_docId",
        "_valueId",
        "_rev",
        "_created",
        "_deleted",
        "lifeArea",
        "name",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_created",
    ]

    export_dataframe(
        pathlib.Path("values"),
        dataframe_format_export(
            df_documents.loc[df_documents["_type"] == "value"],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Values Inventory

In [ ]:
def export_analysis_values_inventories():
    # Documentation of this analysis.
    export_markdown(
        pathlib.Path("valuesInventories"),
        documentation_as_markdown("Values Inventory"),
    )

    # Preliminary documents.
    export_dataframe(
        pathlib.Path(
            "data",
            "valuesInventories.raw",
        ),
        dataframe_format_export(
            df_documents_raw.loc[
                df_documents_raw["_type"] == "valuesInventory"
            ],
            drop_empty_columns=True,
        ),
    )

    export_dataframe(
        pathlib.Path(
            "data",
            "valuesInventories.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "valuesInventory"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted values inventory documents.
    drop_columns = [
        "_set_id",
    ]
    rename_columns = {
        "_type": "_docType",
        "_id": "_docId",
    }
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_docId",
        "_rev",
        "_created",
        "assigned",
        "assignedDateTime",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_rev",
    ]

    export_dataframe(
        pathlib.Path("valuesInventories"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_type"] == "valuesInventory"
            ],
            drop_empty_columns=True,
            drop_columns=drop_columns,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )

### Analysis: Timeline

In [ ]:
def export_analysis_timeline():
    # # Documentation of this analysis.
    # export_markdown(
    #     pathlib.Path("timeline"),
    #     documentation_as_markdown("Timeline"),
    # )

    # Timeline events are derived only. There is no raw export.
    export_dataframe(
        pathlib.Path(
            "data",
            "timeline.transformed",
        ),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_docType"] == "timelineEvent"
            ],
            drop_empty_columns=True,
        ),
    )

    # Formatted timeline event rows.
    rename_columns = {}
    sort_columns = [
        "_docType",
        "recordId",
        "_patientId",
        "_timelineDate",
        "_timelineEvent",
    ]
    sort_rows_by_columns = [
        "recordId",
        "_patientId",
        "_timelineDate",
    ]

    export_dataframe(
        pathlib.Path("timeline"),
        dataframe_format_export(
            df_documents.loc[
                df_documents["_docType"] == "timelineEvent"
            ],
            drop_empty_columns=True,
            rename_columns=rename_columns,
            sort_columns=sort_columns,
            sort_rows_by_columns=sort_rows_by_columns,
        ),
    )



### Execute Exports

Runs all analysis export functions defined above.

In [ ]:
export_analysis_activities()
export_analysis_activity_logs()
export_analysis_activity_schedules()
export_analysis_assessments()
export_analysis_assessment_logs()
export_analysis_case_reviews()
export_analysis_clinical_histories()
export_analysis_mood_logs()
export_analysis_patient_profiles()
export_analysis_review_marks()
export_analysis_safety_plans()
export_analysis_scheduled_assessments()
export_analysis_scheduled_activities()
export_analysis_sessions()
export_analysis_values()
export_analysis_values_inventories()

export_analysis_timeline()

### Write Archive

In [ ]:
# The export is stored in a single zip file.
with open(
    pathlib.Path(
        archive_dir_path,
        "export_{}.zip".format(archive_suffix),
    ),
    mode="xb",
) as archive_file:
    with pyzipper.AESZipFile(
        archive_file,
        "w",
        compression=pyzipper.ZIP_LZMA,
        encryption=pyzipper.WZ_AES,
    ) as archive_zipfile:
        # Set the password
        archive_zipfile.setpassword(archive_password.encode("utf-8"))

        for file_current in export_file_list:
            if file_current.type in [
                ExportFileType.BYTES,
                ExportFileType.EXCEL,
            ]:
                archive_zipfile.writestr(
                    file_current.path.as_posix(), file_current.bytes
                )
            elif file_current.type in [
                ExportFileType.CSV,
                ExportFileType.MARKDOWN,
            ]:
                assert file_current.text is not None

                archive_zipfile.writestr(
                    file_current.path.as_posix(), file_current.text.encode("utf-8")
                )
            else:
                raise ValueError("Unknown ExportFileType")